# Whisper LoRA 파인튜닝 - 외국인 한국어 발음 인식

## 목표
- Whisper-small 모델을 LoRA로 파인튜닝
- 외국인 발음을 "들리는 대로" 변환하도록 학습
- 적은 데이터(30개)로 효율적인 학습

## 1. 환경 설정 및 라이브러리 설치

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
# 필요한 라이브러리 설치 (최초 1회)
# !pip install --upgrade transformers datasets accelerate peft bitsandbytes
# !pip install --upgrade librosa soundfile jiwer evaluate

In [3]:
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA L40S
GPU Memory: 47.8 GB


## 2. 경로 설정 및 데이터 로드

In [4]:
# 경로 설정 (필요에 따라 수정)
WORK_DIR = "data/들리는json"
AUDIO_DIR = "data/들리는wav"  # 음성 파일 폴더 경로 (수정 필요)
EXCEL_PATH = "data/들리는 대로 변환 정리 문서.xlsx"

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f"{WORK_DIR}/checkpoints", exist_ok=True)

print(f"작업 디렉토리: {WORK_DIR}")

작업 디렉토리: data/들리는json


In [5]:
# !pip install openpyxl

In [6]:
# 엑셀 데이터 로드
df = pd.read_excel(EXCEL_PATH)
print(f"총 샘플 수: {len(df)}")
print(f"\n컬럼: {df.columns.tolist()}")
print(f"\n처음 5개 데이터:")
df.head()

총 샘플 수: 31

컬럼: ['음성 번호', '원본 문장', '들리는 문장']

처음 5개 데이터:


,음성 번호,원본 문장,들리는 문장
0,07961-F-1-VI-B-ATQ021-1768242,제 방이 작은 편이에요. 방 안에 침대하고 책상이 있어요. 그리고 오창도 하나 있고...,제 방이 작근 펴니에요. 방 아네 침대하고 채쌍이 이써요. 그리고 오창도 하나 이꼬...
1,07458-F-1-VI-B-PDT001-1879662,한국어를 잘하려면 무작정 책으로만 공부해서는 안 돼. 제일 중요한 것은 한국인하고 ...,한꾸거를 자라려면 무작청 채그로만 곤부해서는 안 돼. 제일 충요한 거슨 한구긴하꼬 ...
2,01666-F-2-VI-B-SSO001-0386569,"사람들은 행복하게 살기 위해 가족, 친구, 건강, 사랑, 일, 돈 등을 말합니다. ...","사람트른 행보카게 살기 위해 가족, 친구, 건강, 사랑, 일, 돈 등을 마랍니다. ..."
3,01689-F-1-VI-B-ATQ028-0785142,집은 구할 때 가장 중요하게 생각하는 거는 햇빛이 많은 것입니다. 왜냐하면 햇빛이 ...,지븐 구할 때 가장 중요하게 생가카는 거는 해삐치 마는 거십니다. 왜냐하면 해삐치 ...
4,01535-F-3-VI-B-SPT006-0432276,저는 어렸을 때 학교에 지각한 적이 있어요. 그때는 교동이 스트레스 때문에 그리고 ...,저는 어려쓸 때 하꾜에 지가칸 저기 이써요. 그때는 교통이 츨스 때무네 그리고 저는...


In [7]:
# 음성 파일 존재 확인
def find_audio_file(audio_id, audio_dir):
    """음성 번호로 파일 찾기"""
    possible_extensions = ['.wav', '.mp3', '.m4a', '.flac']
    
    for ext in possible_extensions:
        # 정확한 파일명
        path = os.path.join(audio_dir, f"{audio_id}{ext}")
        if os.path.exists(path):
            return path
        
        # 부분 매칭
        for file in os.listdir(audio_dir) if os.path.exists(audio_dir) else []:
            if audio_id in file:
                return os.path.join(audio_dir, file)
    
    return None

# 파일 매칭 확인 (AUDIO_DIR이 존재하면)
if os.path.exists(AUDIO_DIR):
    matched = 0
    for idx, row in df.iterrows():
        audio_path = find_audio_file(row['음성 번호'], AUDIO_DIR)
        if audio_path:
            matched += 1
    print(f"매칭된 음성 파일: {matched}/{len(df)}")
else:
    print(f"⚠️ 음성 폴더가 없습니다: {AUDIO_DIR}")
    print("음성 파일을 업로드한 후 AUDIO_DIR 경로를 수정해주세요.")

매칭된 음성 파일: 30/31


In [8]:
# === 추가할 코드 (데이터 로드 후) ===

from g2pk import G2p
g2p = G2p()

def convert_to_pronunciation(text):
    """
    글자 기반 → 발음 기반 레이블 변환
    논문: G2P를 통해 문맥 의존성을 줄이고 발음 기반 학습 강화
    
    예시:
    - '좋겠다' → '조켇따'
    - '막내' → '망내'
    - '한국어' → '한구거'
    """
    try:
        return g2p(text)
    except:
        return text  # 변환 실패 시 원본 유지

# 기존 '들리는 문장'을 발음 기반으로 변환
df['발음_레이블'] = df['들리는 문장'].apply(convert_to_pronunciation)

# 변환 결과 확인
print("=== G2P 변환 예시 ===")
for idx in range(min(5, len(df))):
    print(f"\n원본: {df['들리는 문장'].iloc[idx][:50]}...")
    print(f"발음: {df['발음_레이블'].iloc[idx][:50]}...")

=== G2P 변환 예시 ===

원본: 제 방이 작근 펴니에요. 방 아네 침대하고 채쌍이 이써요. 그리고 오창도 하나 이꼬 선판도...
발음: 제 방이 작끈 펴니에요. 방 아네 침대하고 채쌍이 이써요. 그리고 오창도 하나 이꼬 선판도...

원본: 한꾸거를 자라려면 무작청 채그로만 곤부해서는 안 돼. 제일 충요한 거슨 한구긴하꼬 마니 대...
발음: 한꾸거를 짜라려면 무작청 채그로만 곤부해서느 난 돼. 제일 충요한 거슨 한구긴하꼬 마니 대...

원본: 사람트른 행보카게 살기 위해 가족, 친구, 건강, 사랑, 일, 돈 등을 마랍니다. 여러부는...
발음: 사람트른 행보카게 살기 위해 가족, 친구, 건강, 사랑, 일, 돈 등을 마람니다. 여러부는...

원본: 지븐 구할 때 가장 중요하게 생가카는 거는 해삐치 마는 거십니다. 왜냐하면 해삐치 마느면 ...
발음: 지븐 구할 때 가장 중요하게 생가카는 거는 해삐치 마는 거심니다. 왜냐하면 해삐치 마느면 ...

원본: 저는 어려쓸 때 하꾜에 지가칸 저기 이써요. 그때는 교통이 츨스 때무네 그리고 저는 느께 ...
발음: 저느 너려쓸 때 하꾜에 지가칸 저기 이써요. 그때는 교통이 츨스 때무네 그리고 저는 느께 ...


## 3. Whisper 모델 및 프로세서 로드

In [9]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_NAME = "openai/whisper-small"

print(f"모델 로딩: {MODEL_NAME}")
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# 한국어 설정 (새로운 방식)
model.generation_config.language = "korean"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None  # 자동 설정되도록

print(f"모델 파라미터 수: {model.num_parameters():,}")
print("모델 로드 완료!")

모델 로딩: openai/whisper-small


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

모델 파라미터 수: 241,734,912
모델 로드 완료!


## 4. LoRA 설정

In [10]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# LoRA 설정
lora_config = LoraConfig(
    r=16,  # LoRA rank (작은 데이터셋에는 8-16 권장)
    lora_alpha=32,  # LoRA alpha
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],  # 적용할 모듈
    lora_dropout=0.3,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

# LoRA 적용
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 확인
model.print_trainable_parameters()

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


## 5. 데이터셋 준비

In [11]:
import librosa
from torch.utils.data import Dataset

class KoreanPronunciationDataset(Dataset):
    def __init__(self, dataframe, audio_dir, processor, target_sr=16000):
        self.df = dataframe
        self.audio_dir = audio_dir
        self.processor = processor
        self.target_sr = target_sr
        
        # 유효한 샘플만 필터링
        self.valid_samples = []
        for idx, row in self.df.iterrows():
            audio_path = find_audio_file(row['음성 번호'], audio_dir)
            if audio_path:
                self.valid_samples.append({
                    'audio_path': audio_path,
                    'transcript': row['발음_레이블']
                })
        
        print(f"유효한 샘플 수: {len(self.valid_samples)}")
    
    def __len__(self):
        return len(self.valid_samples)
    
    def __getitem__(self, idx):
        sample = self.valid_samples[idx]
        
        # 오디오 로드
        audio, sr = librosa.load(sample['audio_path'], sr=self.target_sr)
        
        # 프로세서로 입력 특성 추출
        input_features = self.processor(
            audio, 
            sampling_rate=self.target_sr, 
            return_tensors="pt"
        ).input_features.squeeze(0)
        
        # 라벨 토큰화
        labels = self.processor.tokenizer(
            sample['transcript'],
            return_tensors="pt"
        ).input_ids.squeeze(0)
        
        return {
            "input_features": input_features,
            "labels": labels
        }

## 6. Data Collator 정의

In [12]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 입력 특성 패딩
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        
        # 라벨 패딩
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        # 패딩 토큰을 -100으로 변경 (loss 계산에서 무시)
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        
        # BOS 토큰 제거 (있다면)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        
        batch["labels"] = labels
        
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 7. 평가 메트릭 정의

In [13]:
import evaluate
from jiwer import wer, cer

# G2P 초기화
try:
    from g2pk import G2p
    g2p = G2p()
    USE_G2P = True
    print("G2P 로드 완료")
except:
    print("G2P를 사용할 수 없습니다.")
    USE_G2P = False

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    
    # -100을 패드 토큰으로 변환
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    # 디코딩
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    # 🔥 G2P 변환 적용 (예측값과 레이블 모두)
    if USE_G2P:
        pred_str_g2p = [g2p(text) for text in pred_str]
        label_str_g2p = [g2p(text) for text in label_str]
    else:
        pred_str_g2p = pred_str
        label_str_g2p = label_str
    
    # WER, CER 계산 (G2P 적용된 버전)
    wer_score = wer(label_str_g2p, pred_str_g2p)
    cer_score = cer(label_str_g2p, pred_str_g2p)
    
    # PER 계산
    if USE_G2P:
        per_score = cer(label_str_g2p, pred_str_g2p)  # G2P 적용 후 CER = PER
    else:
        per_score = cer_score
    
    results = {
        "wer": wer_score * 100,
        "cer": cer_score * 100,
        "per": per_score * 100,
    }
    
    return results

print("평가 메트릭 함수 정의 완료 (G2P 일관성 적용)")

G2P 로드 완료
평가 메트릭 함수 정의 완료 (G2P 일관성 적용)


## 8. 학습 설정

In [14]:
# 데이터셋 생성
dataset = KoreanPronunciationDataset(df, AUDIO_DIR, processor)

# 🔥 데이터셋 동작 확인
print("\n=== 데이터셋 테스트 ===")
sample = dataset[0]
print(f"Keys: {sample.keys()}")
print(f"input_features shape: {sample['input_features'].shape}")
print(f"labels shape: {sample['labels'].shape}")

# 학습/검증 분리
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# 🔥 split 후에도 동작 확인
print(f"\n=== Split 후 테스트 ===")
train_sample = train_dataset[0]
print(f"Train sample keys: {train_sample.keys()}")

print(f"\n학습 데이터: {len(train_dataset)}")
print(f"검증 데이터: {len(val_dataset)}")

유효한 샘플 수: 30

=== 데이터셋 테스트 ===
Keys: dict_keys(['input_features', 'labels'])
input_features shape: torch.Size([80, 3000])
labels shape: torch.Size([86])

=== Split 후 테스트 ===
Train sample keys: dict_keys(['input_features', 'labels'])

학습 데이터: 27
검증 데이터: 3


In [15]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir=f"{WORK_DIR}/checkpoints",
    
    # 배치 및 에폭 설정
    per_device_train_batch_size=10,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=30,  # 늘리고 early stopping으로 제어
    
    # 학습률 설정
    learning_rate=1e-4,
    warmup_steps=50,
    
    # 평가 및 저장
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="per",  # 🔥 PER 기준으로 변경
    greater_is_better=False,
    
    # 생성 설정
    predict_with_generate=True,
    generation_max_length=225,
    
    # 최적화
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    
    # 로깅
    logging_dir=f"{WORK_DIR}/logs",
    logging_steps=10,
    report_to="none",
)

print("학습 설정 완료 (Early Stopping 적용)")
print("학습 설정 완료")
print(f"총 에폭: {training_args.num_train_epochs}")
print(f"배치 크기: {training_args.per_device_train_batch_size}")
print(f"학습률: {training_args.learning_rate}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


학습 설정 완료 (Early Stopping 적용)
학습 설정 완료
총 에폭: 30
배치 크기: 10
학습률: 0.0001


## 9. Trainer 초기화 및 학습

In [16]:
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]  # 🔥 5 epoch 동안 개선 없으면 중단
)

print("Trainer 초기화 완료 (Early Stopping: patience=5)")

Trainer 초기화 완료 (Early Stopping: patience=5)


In [17]:
# 학습 시작
print("="*60)
print("학습 시작!")
print("="*60)

trainer.train()

학습 시작!


Epoch,Training Loss,Validation Loss,Wer,Cer,Per
1,No log,4.864529,35.820896,14.396887,14.396887
2,No log,4.847383,35.820896,14.396887,14.396887
3,No log,4.825642,35.820896,14.396887,14.396887
4,No log,4.747110,35.820896,14.396887,14.396887
5,7.513009,4.638264,35.820896,14.396887,14.396887
6,7.513009,4.508997,35.820896,14.396887,14.396887


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

TrainOutput(global_step=12, training_loss=7.449367682139079, metrics={'train_runtime': 13.4163, 'train_samples_per_second': 60.374, 'train_steps_per_second': 4.472, 'total_flos': 4.757639970816e+16, 'train_loss': 7.449367682139079, 'epoch': 6.0})

## 10. 모델 저장

In [18]:
# LoRA 어댑터 저장
SAVE_PATH = f"{WORK_DIR}/no7_model_whisper-g2p-final"
model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)

print(f"모델 저장 완료: {SAVE_PATH}")

모델 저장 완료: data/들리는json/no7_model_whisper-g2p-final


## 11. 저장된 모델 다시 로드하기

In [19]:
# 나중에 모델 다시 로드하는 방법
from peft import PeftModel

def load_finetuned_model(lora_path, base_model_name="openai/whisper-small"):
    """저장된 LoRA 모델 로드"""
    # 베이스 모델 로드
    base_model = WhisperForConditionalGeneration.from_pretrained(base_model_name)
    processor = WhisperProcessor.from_pretrained(base_model_name)
    
    # LoRA 어댑터 적용
    model = PeftModel.from_pretrained(base_model, lora_path)
    
    # 한국어 설정
    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="korean", task="transcribe")
    
    return model, processor

# 사용 예시
loaded_model, loaded_processor = load_finetuned_model(SAVE_PATH)
print("모델 로드 함수 정의 완료")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

모델 로드 함수 정의 완료
